## **Análise de série de vazões (Xingó)**

### 1 - Leitura dos dados e gerando hidrogramas

In [ ]:
import hidrocomp

In [ ]:
import pandas as pd
import numpy as np
import plotly as py
import plotly.io as pio
import plotly.graph_objects as go

import sys
#sys.path.insert(0, r'C:\Users\arist\OneDrive\Documentos\UFAL\Pesquisas\lib_hidro_comp\HidroComp_forked\hidrocomp')

from files.ons import Ons
from files.ana import Ana
from series.flow import Flow
from series.chuva import Chuva
from series.circular import Circular

In [2]:
file = "ONS_daily_flow.csv"
dados = pd.read_csv(file, index_col=0, parse_dates=True)
dados = pd.DataFrame(dados["XINGO (178)"])

#Filtrando período de análise ()
#dados = dados[(dados.index >= "1931-09-01") & (dados.index <= "1994-08-31")]
dados = dados[dados.index <= "2018-08-31"]

#Convetendo obj pandas para Series Flow
dados = Flow(pd.DataFrame(dados["XINGO (178)"]))

#Gerando hidrograma
fig, data = dados.plot_hydrogram()
py.offline.plot(fig, filename='gráficos/figura1_hidrograma_xingo.html')
pio.show(fig)

#Calculando picos anuais
peaks_max = dados.maximum(station="XINGO (178)")

#Gerando hidrograma destanco os valores de pico
fig, data = peaks_max.plot_hydrogram()
#py.offline.plot(fig, filename='gráficos/histo_max.html')
pio.show(fig)


### **2 - Séries de durações parciais**

In [11]:
(pd.to_datetime('2025-01-02') - pd.to_datetime('2025-01-01')).days

1

#### 2.1 - Separando eventos hidrológicos para calibração da cópula (1931 - 1994)

In [26]:
file = "ONS_daily_flow.csv"
dados = pd.read_csv(file, index_col=0, parse_dates=True)
dados = pd.DataFrame(dados["XINGO (178)"])

#Filtrando período de análise ()
dados = dados[(dados.index >= "1994-08-31")]

dados.describe()

,XINGO (178)
count,8889.000000
mean,1939.208527
std,1596.888786
min,184.260000
25%,824.000000
50%,1270.000000
75%,2617.000000
max,12194.000000


In [60]:
dados[(dados['XINGO (178)'] >= 4033) & (dados.index >= '2004/01/05')].head(30)

,XINGO (178)
Date,
2004-01-18,4326.0
2004-01-20,4244.0
2004-01-21,4822.0
2004-01-22,5892.0
2004-01-23,7155.0
2004-01-24,8069.0
2004-01-25,8847.0
2004-01-26,8853.0
2004-01-27,8996.0


In [24]:
file = "ONS_daily_flow.csv"
dados = pd.read_csv(file, index_col=0, parse_dates=True)
dados = pd.DataFrame(dados["XINGO (178)"])

#Filtrando período de análise ()
#dados = dados[(dados.index >= "1931-09-01") & (dados.index <= "1994-08-31")]

dados = dados[(dados.index >= "1994-08-31")]

#Convetendo obj pandas para Series Flow
dados = Flow(pd.DataFrame(dados["XINGO (178)"]))

station = "XINGO (178)"

#'stationary' or 'events_by_year'
type_threshold = 'stationary'  

#'flood' or 'drought'
type_event = 'flood'

#'media', 'mediana' or 'autocorrelation' 
type_criterion = 'autocorrelation'

#Peaks or percentil
value_threshold = 4033

sdp_calibration = dados.parcial(station, type_threshold, type_event, type_criterion, value_threshold)

#Armazenando dataframe da SDP calculada
df_sdp_calibration = sdp_calibration.peaks
df_sdp_calibration

fig, data = sdp_calibration .plot_hydrogram(title='SDP Calibragem-Xingo')
py.offline.plot(fig, filename='gráficos/figura2_histo_sdp_calibration.html')

'''value_threshold = 4500
end_threshold = 4650
step = 1

while value_threshold < end_threshold:

 
    value_threshold += step
    sdp_validation = dados.parcial(station, type_threshold, type_event, type_criterion, value_threshold)
    df_sdp_calibration = sdp_validation.peaks

    if (len(df_sdp_calibration[df_sdp_calibration.index == '2005-05-02']) == 1) and (df_sdp_calibration[df_sdp_calibration.index == '2005-05-02']['Duration'].iloc[0] == 1):

        print('Funcionou!')
        print('Peak_limiar = ', value_threshold) 
        print('1996-01-17 - Duration = ', df_sdp_calibration[df_sdp_calibration.index == '1996-01-17']['Duration'].iloc[0])
        print('1998-01-03 - Duration = ', df_sdp_calibration[df_sdp_calibration.index == '1998-01-03']['Duration'].iloc[0])
        print('2000-04-03 - Duration = ', df_sdp_calibration[df_sdp_calibration.index == '2000-04-03']['Duration'].iloc[0])
        print('2004-01-18 - Duration = ', df_sdp_calibration[df_sdp_calibration.index == '2000-04-03']['Duration'].iloc[0])


        print('=============================')'''


"value_threshold = 4500\nend_threshold = 4650\nstep = 1\n\nwhile value_threshold < end_threshold:\n\n \n    value_threshold += step\n    sdp_validation = dados.parcial(station, type_threshold, type_event, type_criterion, value_threshold)\n    df_sdp_calibration = sdp_validation.peaks\n\n    if (len(df_sdp_calibration[df_sdp_calibration.index == '2005-05-02']) == 1) and (df_sdp_calibration[df_sdp_calibration.index == '2005-05-02']['Duration'].iloc[0] == 1):\n\n        print('Funcionou!')\n        print('Peak_limiar = ', value_threshold) \n        print('1996-01-17 - Duration = ', df_sdp_calibration[df_sdp_calibration.index == '1996-01-17']['Duration'].iloc[0])\n        print('1998-01-03 - Duration = ', df_sdp_calibration[df_sdp_calibration.index == '1998-01-03']['Duration'].iloc[0])\n        print('2000-04-03 - Duration = ', df_sdp_calibration[df_sdp_calibration.index == '2000-04-03']['Duration'].iloc[0])\n        print('2004-01-18 - Duration = ', df_sdp_calibration[df_sdp_calibration.i

In [38]:
df_sdp_calibration


,Duration,Start,End,peaks
1995-03-02,14,1995-02-22,1995-03-08,5328.00
1996-01-17,35,1995-12-25,1996-01-29,5481.00
1996-12-09,9,1996-12-04,1996-12-13,4729.00
1997-01-24,47,1997-01-06,1997-02-22,7502.00
1997-04-01,44,1997-03-12,1997-04-25,5904.00
1998-01-03,14,1997-12-25,1998-01-08,5297.00
1998-03-09,12,1998-03-03,1998-03-15,4680.00
1999-01-20,2,1999-01-19,1999-01-21,4044.00
1999-03-22,23,1999-03-14,1999-04-06,5985.00
1999-12-22,2,1999-12-21,1999-12-23,4295.00


In [61]:
date1 = '18/01/2004'
date2 = '06/05/2004'
pd.to_datetime(date1, format='%d/%m/%Y') - pd.to_datetime(date2, format='%d/%m/%Y')

Timedelta('-109 days +00:00:00')

In [4]:
fig, data = sdp_calibration .plot_hydrogram(title='SDP Calibragem-Xingo')
py.offline.plot(fig, filename='gráficos/figura2_histo_sdp_calibration.html')

'gráficos/figura2_histo_sdp_calibration.html'

#### 2.2 - Separando eventos hidrológicos para validação da cópula (1995 - 2018)
Dados OBSERVADOS

In [4]:
flow_observed = pd.read_excel(r'C:\Users\arist\OneDrive\Documentos\UFAL\Pesquisas\PIBIC 24-25\lib_clebson\aristides_pibic\xingo_observed.xlsx')

flow_observed.index = flow_observed['Unnamed: 0'].to_list()
flow_observed['XINGO (178)'] = flow_observed['49340080']
flow_observed.drop(columns=['Unnamed: 0', '49340080'], inplace=True)
flow_observed.index.names = ['Data']

flow_observed

,XINGO (178)
Data,
1995-01-01,1771.0
1995-01-02,1769.0
1995-01-03,1902.0
1995-01-04,2360.0
1995-01-05,2349.0
...,...
2018-08-27,622.0
2018-08-28,621.0
2018-08-29,623.0


In [9]:
flow_observed[flow_observed.index == '2004-01-31']

,XINGO (178)
Data,
2004-01-31,7664.0


In [49]:
# ================ CANDIDATO ================
#  value_threshold = 4070 e 4245


In [4]:
#===================== LENDO DADOS ===========================

file = "ONS_daily_flow.csv"
data_validation = pd.read_csv(file, index_col=0, parse_dates=True)
data_validation = pd.DataFrame(data_validation["XINGO (178)"])

data_validation = data_validation[(data_validation.index >= "1994-09-01") & (data_validation.index <= "2018-08-31")]

#Convetendo obj pandas para Series Flow
data_validation= Flow(pd.DataFrame(data_validation))

#===================== SDPs ===========================
station = "XINGO (178)"

#'stationary' or 'events_by_year'
type_threshold = 'stationary' 

#'flood' or 'drought'
type_event = 'flood'

#'media', 'mediana' or 'autocorrelation' 
type_criterion = 'autocorrelation'
value_threshold = 4033            #4250

sdp_validation = data_validation.parcial(station, type_threshold, type_event, type_criterion, value_threshold)
df_sdp_validation = sdp_validation.peaks

fig, data = sdp_validation.plot_hydrogram(title='SDP 2 - Xingo')
py.offline.plot(fig, filename='gráficos/figura3_histo_sdp_2.html')

'''#============= ITERANDO PARA ENCONTRAR LIMIAR DE PICO ============
value_threshold = 4000
end_threshold = 4400
step = 1

while value_threshold < end_threshold:
    #Peaks or percentil
    #value_threshold = df_sdp_calibration['peaks'].min() #Adotando limiar dos eventos de calibragem
 
    value_threshold += step
    sdp_validation = data_validation.parcial(station, type_threshold, type_event, type_criterion, value_threshold)
    df_sdp_validation = sdp_validation.peaks

    if (len(df_sdp_validation[df_sdp_validation.index == '2004-01-18']) == 1):

        print('Funcionou!')
        print('Peak_limiar = ', value_threshold) 
        print('1996-01-17 - Duration = ', df_sdp_validation[df_sdp_validation.index == '1996-01-17']['Duration'].iloc[0])
        print('1998-01-03 - Duration = ', df_sdp_validation[df_sdp_validation.index == '1998-01-03']['Duration'].iloc[0])
        print('2000-04-03 - Duration = ', df_sdp_validation[df_sdp_validation.index == '2000-04-03']['Duration'].iloc[0])

        print('=============================')'''


"#============= ITERANDO PARA ENCONTRAR LIMIAR DE PICO ============\nvalue_threshold = 4000\nend_threshold = 4400\nstep = 1\n\nwhile value_threshold < end_threshold:\n    #Peaks or percentil\n    #value_threshold = df_sdp_calibration['peaks'].min() #Adotando limiar dos eventos de calibragem\n \n    value_threshold += step\n    sdp_validation = data_validation.parcial(station, type_threshold, type_event, type_criterion, value_threshold)\n    df_sdp_validation = sdp_validation.peaks\n\n    if (len(df_sdp_validation[df_sdp_validation.index == '2004-01-18']) == 1):\n\n        print('Funcionou!')\n        print('Peak_limiar = ', value_threshold) \n        print('1996-01-17 - Duration = ', df_sdp_validation[df_sdp_validation.index == '1996-01-17']['Duration'].iloc[0])\n        print('1998-01-03 - Duration = ', df_sdp_validation[df_sdp_validation.index == '1998-01-03']['Duration'].iloc[0])\n        print('2000-04-03 - Duration = ', df_sdp_validation[df_sdp_validation.index == '2000-04-03']['D

In [5]:
df_sdp_validation


,Duration,Start,End,peaks
1995-03-02,14,1995-02-22,1995-03-08,5328.00
1996-01-17,35,1995-12-25,1996-01-29,5481.00
1996-12-09,9,1996-12-04,1996-12-13,4729.00
1997-01-24,47,1997-01-06,1997-02-22,7502.00
1997-04-01,44,1997-03-12,1997-04-25,5904.00
1998-01-03,14,1997-12-25,1998-01-08,5297.00
1998-03-09,12,1998-03-03,1998-03-15,4680.00
1999-01-20,2,1999-01-19,1999-01-21,4044.00
1999-03-22,23,1999-03-14,1999-04-06,5985.00
1999-12-22,2,1999-12-21,1999-12-23,4295.00


In [6]:
fig, data = sdp_validation .plot_hydrogram(title='SDP-Xingo')
py.offline.plot(fig, filename='gráficos/histo_sdp_validation.html')

'gráficos/histo_sdp_validation.html'

### **3 - Modelagem das distribuições marginais**

#### 3.1 - Magnitude (Generalizada de Pareto)

In [7]:
#Distribuição cumulativa
title = 'Xingó'
type_function = 'cumulative'
fig_acum_mml_pareto, data_acum_mml_pareto = sdp_calibration.plot_distribution(title, type_function, estimador='mml', distribuition='GP', variable='peaks')
py.offline.plot(fig_acum_mml_pareto, filename='gráficos/figura4a_dist_cum_peaks_mml.html')
pio.show(fig_acum_mml_pareto)

#Distribuição densidade-probabilidade
title = 'Xingó'
type_function = 'density'
fig_density_mml_pareto, data_density_mmml_pareto = sdp_calibration.plot_distribution(title, type_function, estimador='mml', distribuition='GP', variable='peaks')
py.offline.plot(fig_density_mml_pareto, filename='gráficos/figura4b_dist_density_peaks_mml.html')
pio.show(fig_density_mml_pareto)

#### 3.2 - Duração ( Pearson III) 

In [7]:
#Distribuição densidade-probabilidade
title = 'Xingó'
type_function = 'density'
fig_acum_mml_pearson, data_acum_mmml_pareto = sdp_calibration.plot_distribution(title, type_function, estimador='mml', distribuition='P3', variable='Duration')
py.offline.plot(fig_acum_mml_pearson, filename='gráficos/figura5a_dist_cum_duration_mml.html')
pio.show(fig_acum_mml_pearson)

#Distribuição densidade-probabilidade
title = 'Xingó'
type_function = 'cumulative'
fig_density_mml_pearson, data_density_mmml_pareto = sdp_calibration.plot_distribution(title, type_function, estimador='mml', distribuition='P3', variable='Duration')
py.offline.plot(fig_density_mml_pearson, filename='gráficos/figura5b_dist_density_duration_mml.html')
pio.show(fig_density_mml_pearson)


#### 3.3 - Período de ocorrência (Von-Mises)

In [7]:
#Criando objetivo para manipular os dados de período de ocorrência de cheias 
period_calibration = Circular(df_sdp_calibration)

#Visualizando gráfico de barras circular
period_calibration.plot_bar_circular(month_num_start_year_hydrologic=9, unit='degrees')

#Gráfico de dispersão
period_calibration.plot_scatter_circular(month_num_start_year_hydrologic=9, unit='degrees')

In [33]:
from scipy.stats import vonmises_line

circular_dates = period_calibration.circular_date(9, 'rad')

vonmises_line.fit(circular_dates)

(0.553914260906888, 2.976129335223642, 0.4992439586508519)

In [204]:
from lmoments3 import distr


In [34]:
circular_dates

1932-02-09    2.763915
1932-12-29    2.048491
1933-02-20    2.960843
1934-02-07    2.737059
1935-02-24    3.029700
                ...   
1992-05-04    4.223125
1993-01-19    2.409989
1993-03-04    3.167414
1994-02-11    2.805916
1994-04-04    3.701054
Name: date_peaks, Length: 138, dtype: float64

#### 3.4 Parâmetros das distribuições marginais

In [42]:
#Salvando série de calibração
calibration_save = pd.DataFrame(sdp_calibration.peaks)
calibration_save['Circular'] = circular_dates

calibration_save.to_excel('sdp_calibration.xlsx')

In [9]:
from scipy.stats import vonmises_line

#Calculando parâmetros (forma, posição e escala)
peaks_GP_params = sdp_calibration.mml('GP', 'peaks')
duration_P3_params = sdp_calibration.mml('P3', 'Duration')

data_degrees = period_calibration.circular_date(9, 'degrees')
period_vonmises_params = vonmises_line.fit(data_degrees)

#Visualizando
print(f'Peaks - GP  {peaks_GP_params}')
print('Duration - P3 ', duration_P3_params)
print('Period - VM ', period_vonmises_params)

Peaks - GP  [0.18351484675917457, 4136.848095908677, 1875.850753029042]
Duration - P3  [2.1681368826691747, 42.71739130434783, 44.917897439365966]
Period - VM  (0.553914260906888, 170.51965019338923, 28.60457177809773)


### **4 - Modelagem função cópula Gaussiana (abordagem frequencista)**

In [7]:
correlation_matrix = np.array([
    [1.0, 0.941, 0.232],
    [0.941, 1.0, 0.155],
    [0.232, 0.155, 1.0]
])

#### 4.1 Calibração das cópulas (1931 - 1994)

In [8]:
from copulae import GaussianCopula
from scipy.stats import genpareto, pearson3, vonmises_line, norm, multivariate_normal, vonmises

##### 4.1.1 Cópula trivariada (Pico, Duração e Período)

In [9]:
#Adicionando ao dataframe os dados circulares
df_sdp_calibration['circular_period'] = period_calibration.circular_date(9, 'degrees')

#Selecionando amostras
X1_calibration = df_sdp_calibration.peaks.to_list()
X2_calibration = df_sdp_calibration.Duration.to_list()
X3_calibration = df_sdp_calibration.circular_period.to_list()

#Transformando distribuição marginais para o espaço uniforme (por meio da CDF)
u1_calibration = genpareto.cdf(X1_calibration , peaks_GP_params[0], peaks_GP_params[1], peaks_GP_params[2])
u2_calibration = pearson3.cdf(X2_calibration , duration_P3_params[0], duration_P3_params[1], duration_P3_params[2])
u3_calibration = vonmises_line.cdf(X3_calibration, period_vonmises_params[0], period_vonmises_params[1], period_vonmises_params[2])

#Transformando distribuição marginais para o espaço uniforme (por meio da CDF)
#u1_calibration = genpareto.cdf(X1_calibration, 0.19, 4112.91,  1835.72)
#u2_calibration = pearson3.cdf(X2_calibration, 2.25, 40.5, 44.57)
#u3_calibration = vonmises_line.cdf(X3_calibration, 0.56, 170.67, 28.58)

#Guardando valores para calibração
data_calibration = pd.DataFrame()
data_calibration['peaks'], data_calibration['duration'], data_calibration['period'] = u1_calibration, u2_calibration, u3_calibration

#Extraindo parâmetros cópula
copule_gaussian_tri = GaussianCopula(dim=3)
copule_gaussian_tri.fit(data_calibration, method='irho')

#Definindo relação de dependência entre as variáveis
#copule_gaussian_tri.params = np.array([0.787, 0.164, 0.117])

#copule_gaussian_tri.params = correlation_matrix
copule_gaussian_tri.summary()

            peaks    duration      period
count  138.000000  138.000000  138.000000
mean     0.500000    0.500000    0.500000
std      0.287626    0.287579    0.287620
min      0.021583    0.021583    0.007194
25%      0.255396    0.258993    0.254496
50%      0.500000    0.500000    0.500000
75%      0.746403    0.746403    0.741007
max      0.992806    0.992806    0.992806


1.000000,0.905373,0.215746
0.905373,1.000000,0.069576
0.215746,0.069576,1.000000


In [296]:
stats.spearmanr(data_calibration['duration'], data_calibration['peaks'])


SignificanceResult(statistic=0.8972061661710694, pvalue=4.0389555560269445e-50)

##### 4.1.2 Cópula bivariada (Duração e Período)

In [10]:
#Extraindo parâmetros cópula bivariada
copule_gaussian_bi = GaussianCopula(dim=2)
#copule_gaussian_bi.fit(data_calibration[['duration', 'period']])

copule_gaussian_bi.params = np.array([0.102176])


copule_gaussian_bi.summary()

1.000000,0.102176
0.102176,1.000000


#### 4.2 Separando dados validação das cópulas (1994 - 2018)

In [11]:
#Criando objetivo para manipular os dados de período de ocorrência de cheias 
period_validation = Circular(df_sdp_validation)

#Calculando datas circuales
df_sdp_validation['circular_period'] = period_validation.circular_date(9, 'degrees')

#Selecionando amostras
X1_validation = df_sdp_validation.peaks.to_list()
X2_validation = df_sdp_validation.Duration.to_list()
X3_validation = df_sdp_validation.circular_period.to_list()

#Transformando distribuição marginais para o espaço uniforme (por meio da CDF)
u1_validation = genpareto.cdf(X1_validation , peaks_GP_params[0], peaks_GP_params[1], peaks_GP_params[2])
u2_validation = pearson3.cdf(X2_validation , duration_P3_params[0], duration_P3_params[1], duration_P3_params[2])
u3_validation = vonmises_line.cdf(X3_validation, period_vonmises_params[0], period_vonmises_params[1],period_vonmises_params[2])


#u1_validation = genpareto.cdf(X1_validation, 0.19, 4112.91,  1835.72)
#u2_validation = pearson3.cdf(X2_validation, 2.25, 40.5, 44.57)
#u3_validation = vonmises_line.cdf(X3_validation, 0.56, 170.67, 28.58)


#Guardando valores para calibração
data_validation = pd.DataFrame()
data_validation['peaks'], data_validation['duration'], data_validation['period'] = u1_validation, u2_validation, u3_validation

#### 4.4 Tempo de retorno multivariado (Picos como variável de referência)

In [12]:
#Calculando CDF multivariadas
cdf_copule_tri = copule_gaussian_tri.cdf(data_validation)
cdf_copule_bi = copule_gaussian_bi.cdf(data_validation[['duration', 'period']])

#Calculando mu
## quantidade de eventos observados e intervalo (quantidade) de anos em que esses eventos ocorreram
n_events = len(df_sdp_validation)
n_years = df_sdp_validation.index.year.max() - df_sdp_validation.index.year.min()

mu = n_events/n_years

#Calculando tempo de retorno multivariado
tr_copule_peaks = mu / (1 - (cdf_copule_tri/cdf_copule_bi))

#Calculando tempo de retorno univariado (pico)
tr_univariate_peaks = mu / (1 - u1_validation)

#Salvando e visualizando resultados
df_sdp_validation['tr_mult'] = tr_copule_peaks
df_sdp_validation['tr_uni'] = tr_univariate_peaks 
df_sdp_validation

,Duration,Start,End,peaks,circular_period,tr_mult,tr_uni
1995-03-02,14,1995-02-22,1995-03-08,5328.00,179.506849,34.084059,3.299276
1996-01-17,35,1995-12-25,1996-01-29,5481.00,135.737705,13.499473,3.547588
1996-12-09,9,1996-12-04,1996-12-13,4729.00,97.643836,16.087697,2.459431
1997-01-24,47,1997-01-06,1997-02-22,7502.00,143.013699,66.343566,8.531995
1997-04-01,44,1997-03-12,1997-04-25,5904.00,209.095890,12.365907,4.314586
1998-01-03,14,1997-12-25,1998-01-08,5297.00,122.301370,27.134371,3.250737
1998-03-09,12,1998-03-03,1998-03-15,4680.00,186.410959,8.561207,2.399313
1999-01-20,2,1999-01-19,1999-01-21,4044.00,139.068493,1.809524,1.809524
1999-03-22,23,1999-03-14,1999-04-06,5985.00,199.232877,47.664785,4.475836
1999-12-22,2,1999-12-21,1999-12-23,4295.00,110.163934,13.697048,1.967429


In [14]:
df_sdp_validation.to_excel('resultados_tr.xlsx')

In [13]:
#Calculando CDF multivariadas
cdf_copule_tri = copule_gaussian_tri.cdf(data_validation)
cdf_copule_bi = copule_gaussian_bi.cdf(data_validation[['duration', 'period']])

#Calculando mu
## quantidade de eventos observados e intervalo (quantidade) de anos em que esses eventos ocorreram
n_events = len(df_sdp_validation)
n_years = df_sdp_validation.index.year.max() - df_sdp_validation.index.year.min()

#mu = n_events/n_years
mu = 1.65

#Calculando tempo de retorno multivariado
tr_copule_peaks = mu / (1 - (cdf_copule_tri/cdf_copule_bi))

#Calculando tempo de retorno univariado (pico)
tr_univariate_peaks = mu / (1 - u1_validation)

#Salvando e visualizando resultados
df_sdp_validation['tr_mult'] = tr_copule_peaks
df_sdp_validation['tr_uni'] = tr_univariate_peaks 
df_sdp_validation

,Duration,Start,End,peaks,circular_period,tr_mult,tr_uni
1995-03-02,14,1995-02-22,1995-03-08,5328.00,179.506849,31.063019,3.008419
1996-01-17,35,1995-12-25,1996-01-29,5481.00,135.737705,12.309059,3.234840
1996-12-09,9,1996-12-04,1996-12-13,4729.00,97.643836,14.690977,2.242612
1997-01-24,47,1997-01-06,1997-02-22,7502.00,143.013699,60.518580,7.779832
1997-04-01,44,1997-03-12,1997-04-25,5904.00,209.095890,11.274705,3.934221
1998-01-03,14,1997-12-25,1998-01-08,5297.00,122.301370,24.744467,2.964159
1998-03-09,12,1998-03-03,1998-03-15,4680.00,186.410959,7.806235,2.187795
1999-01-20,2,1999-01-19,1999-01-21,4044.00,139.068493,1.650000,1.650000
1999-03-22,23,1999-03-14,1999-04-06,5985.00,199.232877,43.449648,4.081256
1999-12-22,2,1999-12-21,1999-12-23,4295.00,110.163934,12.478729,1.793985


## 5. Considerações sobre estacionariedadae das vazões

## 6. Cópula Bayesiana

In [16]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import pymc as pm

from pymc import HalfCauchy, Model, Normal, sample

print(f"Running on PyMC v{pm.__version__}")

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


Running on PyMC v5.21.1


In [48]:
# -----------------------------
# Dados marginais observados
# -----------------------------
x1 = np.array(X1_validation)  # Generalizada de Pareto
x2 = np.array(X2_validation)  # Pearson III
x3 = np.array(X3_validation)  # Von Mises
x = np.column_stack([x1, x2, x3])
n, d = x.shape

# -----------------------------
# Matriz de correlação estimada previamente
# -----------------------------
R = np.array([
    [1.0, 0.941, 0.232],
    [0.941, 1.0, 0.155],
    [0.232, 0.155, 1.0]
])

In [ ]:
'''Peaks - GP  [0.18351484675917457, 4136.848095908677, 1875.850753029042]
Duration - P3  [2.1681368826691747, 42.71739130434783, 44.917897439365966]
Period - VM  (0.553914260906888, 170.51965019338923, 28.60457177809773)'''

In [61]:
import pymc as pm
import numpy as np
import aesara.tensor as at
from scipy.stats import genpareto, gamma, vonmises

# Garantir formato correto
x1 = np.asarray(x1).flatten().astype(np.float64)
x2 = np.asarray(x2).flatten().astype(np.float64)
x3 = np.asarray(x3).flatten().astype(np.float64)

with pm.Model() as model:
    # ----- x1: Generalized Pareto -----
    xi1 = pm.Normal("xi1", mu=0, sigma=0.1)
    loc1 = pm.Uniform("loc1", lower=4000, upper=4200)
    sigma1 = pm.Uniform("sigma1", lower=1700, upper=1900)

    def gpd_logp(x):
        def logp_fn(xi, sigma, loc):
            return at.sum(genpareto.logpdf(x, c=xi, scale=sigma, loc=loc))
        return logp_fn

    pm.DensityDist(
        "x1_obs",
        logp=gpd_logp(x1),
        inputs=[xi1, sigma1, loc1],
        observed=x1,
    )

    # ----- x2: Pearson III (Gamma deslocada) -----
    alpha2 = pm.Normal("alpha2", mu=0, sigma=3)
    beta2 = pm.Uniform("beta2", lower=0.1, upper=20)
    loc2 = pm.Uniform("loc2", lower=-10, upper=10)

    def gamma_logp(x):
        def logp_fn(alpha, beta, loc):
            return at.sum(gamma.logpdf(x, a=alpha, scale=1 / beta, loc=loc))
        return logp_fn

    pm.DensityDist(
        "x2_obs",
        logp=gamma_logp(x2),
        inputs=[alpha2, beta2, loc2],
        observed=x2,
    )

    # ----- x3: von Mises -----
    kappa3 = pm.Normal("kappa3", mu=0, sigma=3)
    mu3 = pm.Uniform("mu3", lower=-np.pi, upper=np.pi)

    def vonmises_logp(x):
        def logp_fn(kappa, mu):
            return at.sum(vonmises.logpdf(x, kappa=kappa, loc=mu))
        return logp_fn

    pm.DensityDist(
        "x3_obs",
        logp=vonmises_logp(x3),
        inputs=[kappa3, mu3],
        observed=x3,
    )

    # ----- Amostragem -----
    trace = pm.sample(1000, tune=1000, target_accept=0.9)



ModuleNotFoundError: No module named 'numpy.distutils'

In [57]:
pip install numpy==1.25.2


     ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/10.8 MB 6.7 MB/s eta 0:00:02
     ---------- ----------------------------- 2.9/10.8 MB 8.0 MB/s eta 0:00:01
     ------------------- -------------------- 5.2/10.8 MB 9.4 MB/s eta 0:00:01
     --------------------------- ------------ 7.3/10.8 MB 9.3 MB/s eta 0:00:01
     -------------------------------- ------- 8.9/10.8 MB 8.9 MB/s eta 0:00:01
     -------------------------------------- - 10.5/10.8 MB 8.6 MB/s eta 0:00:01
     ---------------------------------------- 10.8/10.8 MB 8.3 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [33 lines of output]
      Traceback (most recent call last):
        File "c:\Users\arist\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 353, in <module>
          main()
        File "c:\Users\arist\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 335, in main
          json_out['return_val'] = hook(**hook_input['kwargs'])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "c:\Users\arist\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 112, in get_requires_for_build_wheel
          backend = _build_backend()
                    ^^^^^^^^^^^^^^^^
        File "c:\Users\arist\AppData\Local\Programs\Python\Python312\Lib\si

In [51]:
print(x1[:5])
print(x2[:5])
print(x3[:5])


[5328. 5481. 4729. 7502. 5904.]
[14. 35.  9. 47. 44.]
[179.50684932 135.73770492  97.64383562 143.01369863 209.09589041]


In [46]:
print("x1:", type(x1), x1.shape, x1.dtype)
print("x2:", type(x2), x2.shape, x2.dtype)
print("x3:", type(x3), x3.shape, x3.dtype)



x1: <class 'numpy.ndarray'> (38,) float64
x2: <class 'numpy.ndarray'> (38,) float64
x3: <class 'numpy.ndarray'> (38,) float64


In [ ]:
#Pegar uma amostra da posterior
xi1_, sigma1_, loc1_ = posterior_samples["xi1"].values[0], posterior_samples["sigma1"].values[0], posterior_samples["loc1"].values[0]
alpha2_, beta2_, loc2_ = posterior_samples["alpha2"].values[0], posterior_samples["beta2"].values[0], posterior_samples["loc2"].values[0]
mu3_, kappa3_ = posterior_samples["mu3"].values[0], posterior_samples["kappa3"].values[0]

# Transformar para uniforme via CDF
u1 = genpareto.cdf(x1, c=xi1_, loc=loc1_, scale=sigma1_)
u2 = gamma.cdf(x2 - loc2_, a=alpha2_, scale=1 / beta2_)
u3 = vonmises.cdf(x3, kappa=kappa3_, loc=mu3_)
u = np.column_stack([u1, u2, u3])

# Aplicar a cópula
cop = GaussianCopula(dim=3)
cop.corr = R

# Avaliar densidade conjunta (opcional)
log_lik = cop.log_pdf(u)

# Simulações conjuntas
u_sim = cop.random(n=1000)
x1_sim = genpareto.ppf(u_sim[:, 0], c=xi1_, loc=loc1_, scale=sigma1_)
x2_sim = gamma.ppf(u_sim[:, 1], a=alpha2_, scale=1 / beta2_) + loc2_
x3_sim = vonmises.ppf(u_sim[:, 2], kappa=kappa3_, loc=mu3_)
